# 2.3.1 — Word count distribuito sul full text di CORD-19

Questo notebook **importa** `word_count.py`: non riscrive l'algoritmo e non lancia
sottoprocessi. Tutta la logica sta nel modulo, qui c'è l'esecuzione e la lettura dei
risultati, così esiste una sola sorgente di verità.

L'algoritmo è quello dell'assignment (§2.3.1):

| fase | cosa produce |
|---|---|
| **Map** | per ogni documento *D*, le coppie `(w, cp(w))` — quante volte la parola `w` compare in *D* |
| **Reduce** | per ogni parola `w`, `c(w) = Σ cp(w)` su tutti i documenti |

Struttura dati: **Bag**, come raccomanda il testo («we recommend utilizing the RDD/Bag
data structure»).

Input: `data/silver/paragraphs` — una riga per paragrafo, già sanificato dalla pipeline
di conversione (vedi `DATA_DICTIONARY.md`).

Le scelte di pulizia del testo sono tutte motivate da misure sul corpus: la
giustificazione riga per riga è nel README di questa cartella.

## 1 · Cluster

Dove gira il calcolo lo decide `cluster.txt` alla root del repo (git-ignored), non il
codice: sul Mac parte un `LocalCluster`, sulla VM un `SSHCluster` sui nodi elencati.
Lo stesso notebook gira nei due posti senza modifiche.

In [ ]:
import sys
import time
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "Giulia" else Path.cwd()
sys.path.insert(0, str(REPO))
sys.path.insert(0, str(REPO / "Giulia"))

from bench import get_client, sweep, unmanaged_memory
import word_count as wc

client, cluster = get_client(repo_root=REPO)
client

## 2 · I dati

`read_paragraphs` legge tre colonne e scarta i paragrafi `is_reference_like`
(dichiarazioni di conflitto d'interesse, author contributions, funding: l'1,24% del
corpus). Il filtro sta sul DataFrame, prima di passare al Bag, perché una maschera
vettoriale su una colonna booleana costa molto meno che testare ogni elemento.

In [ ]:
INPUT = REPO / "data" / "silver" / "paragraphs"

paragraphs = wc.read_paragraphs(INPUT)
print("partitions:", paragraphs.npartitions)
paragraphs.take(1)

## 3 · Come si comporta la sanitizzazione

Un controllo a occhio prima di lanciare il calcolo vero: cosa resta di un testo con
trattini tipografici, lettere greche e stop-word.

In [ ]:
demo = "The SARS–CoV-2 virus and TNF-α were measured at 5 µg/mL in Müller's study."
print(wc.sanitize(demo))
print(wc.words(demo))

## 4 · Il grafo delle due fasi

`word_count` restituisce i due Bag, ancora **lazy**: nulla è stato calcolato.

**La fase Map non fa passare niente in rete.** `doc_counts` ha lo stesso numero di
partizioni dell'input: il conteggio per documento avviene *dentro* la partizione, ed è il
*combiner* del MapReduce classico. Ciò che esce dal worker è una entry per
`(documento, parola)`, non una per occorrenza.

**La fase Reduce ha due formulazioni**, che calcolano la stessa identica cosa
(verificato sul corpus intero: 6.037.808 parole, 785.753.529 occorrenze, ogni conteggio
uguale) ma si comportano in modo molto diverso:

| | `split_out=0` — `Bag.foldby` | `split_out=16` — groupby DataFrame |
|---|---|---|
| partizioni in uscita | **1** | 16 |
| coda del calcolo | un task solo, seriale | 16 task in parallelo |
| memoria del task finale | tutto il vocabolario (~1,5–2 GB) | ~vocabolario/16 |

`Bag.foldby` e `Bag.frequencies` riducono **sempre** a una partizione sola: tutto lo
spazio delle chiavi deve stare in un singolo task, su un singolo worker. Con la parola
come chiave è fattibile — il vocabolario satura al crescere del corpus (legge di
Heaps) — ma resta una coda seriale. Misurato sul corpus intero:

```
7,0 GB/worker,  foldby         1128 s,  0 worker uccisi
3,4 GB/worker,  foldby         1762 s,  3 worker uccisi
3,4 GB/worker,  split_out=16    274 s,  0 worker uccisi
```

Da qui il default. Il testo dell'assignment permette esplicitamente di passare da Bag a
DataFrame, e la fase Map — dove sta il lavoro vero — resta Bag.

In [ ]:
doc_counts, global_counts = wc.word_count(paragraphs)
print("input        partitions:", paragraphs.npartitions)
print("doc_counts   partitions:", doc_counts.npartitions, "  (Map: nessuno shuffle)")
print("global_counts partitions:", global_counts.npartitions)

_, foldby_counts = wc.word_count(paragraphs, split_out=0)
print("con split_out=0        :", foldby_counts.npartitions, "  (foldby: coda seriale)")

## 5 · Smoke test

Prima del run completo, le stesse identiche operazioni su poche partizioni.

In [ ]:
smoke = wc.read_paragraphs(INPUT, npartitions=8)
_, smoke_counts = wc.word_count(smoke)
smoke_counts.topk(10, key=1).compute()

## 6 · Run completo

`topk` pota presto: ogni partizione inoltra solo le sue prime N, quindi non serve
materializzare tutto il vocabolario per avere la classifica.

In [ ]:
TOP_N = 20

started = time.perf_counter()
top = global_counts.topk(TOP_N, key=1).compute()
elapsed = time.perf_counter() - started
print(f"{elapsed:.1f} s")

for word, count in top:
    print(f"{count:>12,}  {word}")

## 7 · Verifica dell'invariante

Il Reduce **raggruppa e basta**: non perde né inventa occorrenze. Il controllo somma i
conteggi prima e dopo e verifica che coincidano.

Nessun totale assoluto scritto a mano: il dump locale e il corpus della VM sono dataset
diversi, quindi l'unica garanzia sensata è strutturale (`PROJECT_CONTEXT.md`, regola 8.2).

Costa: sommare i conteggi per-documento obbliga a percorrere tutta la tabella
intermedia, mentre `topk` può potare. Si lancia in sviluppo e prima di una consegna,
non a ogni esecuzione.

In [ ]:
import dask

after_map, after_reduce = dask.compute(doc_counts.pluck(1).sum(), global_counts.pluck(1).sum())
assert after_map == after_reduce, (after_map, after_reduce)
print(f"invariante ok: {after_map:,} occorrenze prima e dopo il reduce")

## 8 · Barplot

Il grafico che l'assignment chiede esplicitamente («create a barplot of the top
words»).

In [ ]:
OUT = REPO / "reports" / "word_count"
OUT.mkdir(parents=True, exist_ok=True)

wc.barplot(top, OUT / "top_words.png", f"Top {len(top)} words in the CORD-19 body text")

from IPython.display import Image
Image(str(OUT / "top_words.png"))

## 9 · Benchmark obbligatori

Le linee guida del corso li richiedono esplicitamente — tempo di esecuzione contro
**numero di partizioni** e contro **numero di worker** — e senza il progetto è
considerato incompleto.

**I benchmark non si eseguono qui.** Girano con `Giulia/bench_word_count.py`, headless
sul cluster, un blocco per invocazione; questa sezione ne **legge i CSV**. La ragione è
pratica: la campagna dura ore, e una cella che muore a metà non deve portarsi via le
misure già fatte né obbligare a rieseguire il notebook per rivedere un grafico.

```bash
tmux new -s bench
source ~/pyvenv/bin/activate && cd ~/MAPD-Project
for b in A1 A2 A3a A3b A4 A5 A6 D1 D2; do
  python Giulia/bench_word_count.py $b --input ~/mapd-data/silver/paragraphs
done
```

| blocco | cosa misura |
|---|---|
| `A1` | dove va il tempo: sola lettura / fase Map / job completo |
| **`A2`** | **tempo vs numero di partizioni** (obbligatorio) |
| `A3a` | tempo vs strategia di Reduce, payload `topk` |
| `A3b` | le stesse strategie scrivendo il vocabolario — il payload che ha ucciso il cluster |
| `A4` | granularità del combiner su fette crescenti |
| `A5` | tempo vs volume di dati |
| **`A6`** | **tempo vs numero di worker** (obbligatorio) |
| `D1`/`D2` | run di riferimento sul corpus intero |

Le precisazioni che cambiano il significato della misura:

- nello sweep sulle partizioni **i dati restano gli stessi** e cambia solo come sono
  suddivisi. Misurare fette di corpus via via più grandi è un'altra domanda, ed è `A5`;
- il numero di partizioni si controlla **raggruppando i file in lettura**, non con un
  `repartition` a valle. Attraverso `dd.read_parquet` il partizionamento non è una
  manopola ma una proprietà che si scopre: `blocksize` non lo muove (da 4 a 64 MB, con e
  senza Client, sempre 1979 — ogni Parquet del silver ha un solo row-group), e poi
  `to_bag` forza `optimize()`, che **fonde le partizioni piccole**. Su 1979 file:
  `dd.read_parquet(...).npartitions` dice 1979, e quello che gira davvero sono **990**.
  Per un benchmark che ha il partizionamento come variabile indipendente è squalificante;
- **`A6` gira per ultimo**: su `SSHCluster` il pool di worker è la lista di host e
  `scale()` non può risalire. Nel notebook precedente non era così, e lo sweep sulle
  strategie di Reduce finiva per essere misurato su un worker solo;
- `measure` ripete ogni misura, chiama `sweep()` **tra** una ripetizione e l'altra (mai
  dentro una regione cronometrata), non usa `client.restart()`, conserva tutte le
  ripetizioni — è la dispersione a dire se una differenza è reale — e registra worker,
  thread e unmanaged **effettivi** al momento della misura.

Il benchmark va fatto **sul corpus vero**: su una fetta piccola i tempi sono dominati
dall'overhead di scheduling e più worker risultano più lenti di uno solo.

In [ ]:
import pandas as pd
from bench import plot_scaling, plot_speedup

BENCH = REPO / "reports" / "bench"
BLOCKS = ("A1", "A2", "A3a", "A3b", "A4", "A5", "A6", "B", "D1", "D2")

def load(block):
    path = BENCH / f"{block}.csv"
    return pd.read_csv(path) if path.exists() else pd.DataFrame()

data = {block: load(block) for block in BLOCKS}

# Cosa c'e' davvero, e quanto ne e' sopravvissuto. Le misure senza `seconds` non sono
# buchi nella tabella: sono configurazioni che non hanno completato, ed e' un risultato.
for block, frame in data.items():
    if frame.empty:
        print(f"{block:<4} assente")
        continue
    fallite = frame["seconds"].isna().sum()
    print(f"{block:<4} {len(frame):>3} misure, {fallite} non completate, "
          f"worker {sorted(frame['workers'].unique())}")

### 9.0 · Due controlli prima di guardare le curve

Il primo è dove va il tempo (`A1`): se la sola lettura domina, nessuna delle altre curve
parla di calcolo — su questo cluster ogni byte arriva da un solo server NFS.

Il secondo è che le misure siano state prese sulla configurazione che credevamo. Ogni
riga porta i worker **reali** al momento in cui è stata presa: dentro un blocco quella
colonna deve essere costante. È il controllo che avrebbe intercettato subito lo sweep
sulle strategie di Reduce misurato per sbaglio su un worker solo.

In [ ]:
if not data["A1"].empty:
    fasi = data["A1"].groupby("phase")["seconds"].agg(["mean", "min", "max"])
    fasi = fasi.reindex(["read", "map", "full"]).dropna(how="all")
    fasi["quota sul job"] = fasi["mean"] / fasi.loc["full", "mean"]
    display(fasi.style.format({"mean": "{:.1f}", "min": "{:.1f}", "max": "{:.1f}",
                               "quota sul job": "{:.0%}"}))

for block, frame in data.items():
    if frame.empty or block == "A6":     # A6 fa variare i worker: e' il suo mestiere
        continue
    if frame["workers"].nunique() > 1:
        print(f"ATTENZIONE {block}: misurato su {sorted(frame['workers'].unique())} "
              "worker diversi, le righe non sono confrontabili fra loro")

### 9.1 · Tempo vs numero di partizioni *(obbligatorio)*

Stessi dati, solo tagliati in modo diverso. A dato fisso «numero di partizioni» **è**
«taglia della partizione»: pochi task grossi a sinistra, tanti task minuscoli a destra.

Cosa aspettarsi ai due estremi, ed è il motivo per cui il sweep li include entrambi: a
destra il tempo cresce per overhead di scheduling (migliaia di task che fanno pochissimo
lavoro ciascuno); a sinistra il picco di memoria per task cresce con la partizione, e su
worker da 3,5 GB può non essere una questione di lentezza ma di fattibilità — una misura
mancante in fondo a sinistra dice esattamente questo.

In [ ]:
frame = data["A2"]
if not frame.empty:
    plot_scaling(frame.to_dict("records"), "partitions",
                 BENCH / "a2_partizioni.png", "Word count: tempo vs numero di partizioni")
    display(Image(str(BENCH / "a2_partizioni.png")))
    display(frame.groupby(["partitions", "partition_control"])["seconds"]
            .agg(["mean", "std", "count"]).round(1))

### 9.2 · Tempo vs numero di worker *(obbligatorio)*

Partizionamento fisso e dati fissi: cambia solo quanta macchina lavora. Il sweep scende,
perché su `SSHCluster` i worker che si possono avere sono quelli elencati in
`cluster.txt`.

Sotto ai secondi ci sono le due curve che rispondono davvero alla domanda del corso:
**speedup** S(n) = T(1)/T(n) e **efficienza** E(n) = S(n)/n. I secondi dicono che il job
è andato più veloce; S ed E dicono quanto di ciò che è stato aggiunto sta producendo. È
l'efficienza la curva onesta — e su una fetta piccola scende, fino al caso limite già
osservato di un worker che ne batte quattro.

In [ ]:
frame = data["A6"]
if not frame.empty:
    plot_scaling(frame.to_dict("records"), "workers",
                 BENCH / "a6_worker.png", "Word count: tempo vs numero di worker")
    plot_speedup(frame.to_dict("records"),
                 BENCH / "a6_speedup.png", "Word count: speedup ed efficienza")
    display(Image(str(BENCH / "a6_worker.png")), Image(str(BENCH / "a6_speedup.png")))

    tempi = frame.groupby("workers")["seconds"].mean()
    riferimento = tempi.index.min()
    riepilogo = pd.DataFrame({
        "secondi": tempi.round(1),
        "speedup": (tempi[riferimento] / tempi).round(2),
        "efficienza": (tempi[riferimento] / tempi / (tempi.index / riferimento)).round(2),
    })
    display(riepilogo)

### 9.3 · Dove finisce il Reduce

Non è fra i due benchmark obbligatori ed è il confronto più istruttivo dei tre: stesso
risultato, stessi dati, stesso cluster, e una differenza dovuta solo a **dove finisce la
riduzione**.

`Bag.foldby` e `Bag.frequencies` riducono **sempre a una partizione sola**: tutto lo
spazio delle chiavi deve stare in un singolo task, su un singolo worker. Con la parola
come chiave è fattibile — il vocabolario satura al crescere del corpus, legge di Heaps —
ma resta una **coda seriale**. Un `groupby` di DataFrame invece sa spezzare la propria
uscita: `split_out=N` distribuisce le parole per hash su N partizioni.

`split_out=1` è il punto che rende leggibile il confronto: anche lui chiude in **una**
partizione, quindi *foldby contro 1* isola «dizionario Python contro Arrow» e *1 contro
16* isola «coda seriale contro coda parallela». Senza quel punto i due effetti si leggono
come un numero solo.

**`A3b` misura le stesse strategie scrivendo il vocabolario completo**, cioè il job vero.
Non è una rifinitura: l'11 agosto 2026, sul corpus intero, il run è morto proprio lì —
`KilledWorker` sul task `('foldby-b-to_dataframe-…', 0)` mentre scriveva il Parquet, dopo
che il `topk` della riga precedente era passato. Le due formulazioni non differiscono
solo in secondi, differiscono nel fatto che il job esista.

In [ ]:
frame = data["A3a"]
if not frame.empty:
    plot_scaling(frame.to_dict("records"), "split_out",
                 BENCH / "a3_reduce.png", "Word count: tempo vs sharding del Reduce")
    display(Image(str(BENCH / "a3_reduce.png")))

confronto = pd.concat([data["A3a"], data["A3b"]])
if not confronto.empty:
    tabella = (confronto.groupby(["payload", "split_out"])
               .agg(secondi=("seconds", "mean"),
                    completate=("seconds", "count"),
                    tentate=("repeat", "size"),
                    worker_ripartiti=("restarted", "max"))
               .round(1))
    display(tabella)
    print("split_out=0 e' Bag.foldby. `completate` < `tentate` significa che quella "
          "configurazione non ha finito: e' il risultato, non un buco.")

### 9.4 · Quanto costa la granularità del Map

La fase Map emette `((cord_uid, word), cp)`, una entry **per documento**, perché è quello
che chiede l'assignment (§2.3.1: *«for each document D, emit the pairs (w, cp(w))»*). Non
è la formulazione più economica, e qui si misura di quanto:

| | cosa esce dal Map | |
|---|---|---|
| **L0** `none` | una entry per **occorrenza** | nessun combiner: la versione che si scrive per prima e che non regge |
| **L1** `document` | una entry per **(documento, parola)** | quella dell'assignment, il nostro default |
| **L2** `partition` | una entry per **(partizione, parola)** | il minimo che la fase Reduce possa ricevere |

Misurato su fette crescenti e non a una taglia sola, perché la quantità interessante non
è il rapporto fra i tempi: è **dove ciascun gradino smette di completare**. Il combiner
non fa solo risparmiare tempo, sposta la frontiera di ciò che gira.

`map_rows` è il volume che attraversa la rete, ed è la variabile contro cui i tempi vanno
letti.

In [ ]:
frame = data["A4"]
if not frame.empty:
    riepilogo = (frame.groupby(["files", "combiner"])
                 .agg(righe_dal_map=("map_rows", "max"),
                      secondi=("seconds", "mean"),
                      completate=("seconds", "count"))
                 .unstack("combiner").round(1))
    display(riepilogo)

    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(7, 4.5))
    for combiner, gruppo in frame.dropna(subset=["seconds", "map_rows"]).groupby("combiner"):
        medie = gruppo.groupby("map_rows")["seconds"].mean().sort_index()
        ax.plot(medie.index, medie.values, marker="o", label=combiner)
    ax.set_xlabel("righe in uscita dal Map (volume dello shuffle)")
    ax.set_ylabel("secondi")
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_title("Word count: tempo vs volume di shuffle, per granularita' del combiner")
    ax.grid(alpha=0.25, which="both"); ax.legend()
    plt.show()

## 10 · Memoria dei worker

La colonna `unmanaged_gb_max` dei CSV la registra a ogni misura della campagna: se cresce
in modo **lineare** con le ripetizioni c'è un hotspot di churn nel task; se oscilla
attorno a un plateau è solo working set (`docs/MEMORY_LEAK_REPORT.md`, §7.4). È il
massimo **per worker** e non la somma, perché la somma cresce col numero di worker e
quindi non è confrontabile fra i punti dello sweep sui worker — mentre il worker peggiore
è esattamente quello che sbatte contro `memory_limit`.

Qui sotto la stessa cosa sul cluster **vivo** di questo notebook. `sweep` restituisce al
sistema operativo la memoria che i worker trattengono senza più usarla: va chiamata
**tra** le misure cronometrate, mai dentro una.


In [ ]:
for address, unmanaged in unmanaged_memory(client).items():
    print(f"{address:<28} unmanaged {unmanaged / 1e9:5.2f} GB")

sweep(client)

## 11 · Chiusura

In [ ]:
client.close()
if cluster is not None:
    cluster.close()
print("cluster chiuso")